In [1]:
import pandas as pd

In [15]:
train_df = pd.read_csv("churn-bigml-80.csv")   
test_df = pd.read_csv("churn-bigml-20.csv")

In [16]:
train_df.shape

(2666, 20)

In [17]:
test_df.shape

(667, 20)

In [18]:
def preprocess(df):
    df = df.copy()
    df = df.drop(columns = ["State", "Area code"])
    df["International plan"] = df["International plan"].map({"Yes": 1, "No": 0})
    df["Voice mail plan"] = df["Voice mail plan"].map({"Yes": 1, "No": 0})
    df["Churn"] = df["Churn"].astype(int)         
    return df
train_df = preprocess(train_df)
test_df = preprocess(test_df)

In [19]:
X_train = train_df.drop(columns=["Churn"])
y_train = train_df["Churn"]
X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

In [20]:
from sklearn.tree import DecisionTreeClassifier
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [21]:
path = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

best_f1 = 0
best_alpha = 0
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    from sklearn.metrics import f1_score
    f1 = f1_score(y_test, clf.predict(X_test))
    if f1 > best_f1:
        best_f1 = f1
        best_alpha = alpha

dt_pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
dt_pruned.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [22]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = dt_pruned.predict(X_test)
print("Test F1-score:", f1_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print(confusion_matrix(y_test, y_pred))

Test F1-score: 0.8379888268156425

              precision    recall  f1-score   support

    No Churn       0.97      0.98      0.97       572
       Churn       0.89      0.79      0.84        95

    accuracy                           0.96       667
   macro avg       0.93      0.89      0.91       667
weighted avg       0.96      0.96      0.96       667

[[563   9]
 [ 20  75]]
